# Compare Greenwashing Analysis to SFDR
(SFDR = Sustainable Finance Disclosure Regulation)

**Author:**Emanuel Zeder


**Article 8 (SFDR)**

Financial products that promote environmental or social characteristics.
They must disclose how those characteristics are met and ensure **good governance** in the companies they invest in.
(“Light-green” products.)

**Article 9 (SFDR)**

Financial products that have a sustainable investment objective.
They must show how they meet this objective, ensure **do-no-significant-harm**, and follow **good governance**.
(“Dark-green” products.)

Amundi 2021: € 780 bn  out of € 2064 bn (37.79%)

Amundi 2023: € 852.94 bn out of € 2037 bn (41.87%)

Sources:

https://cpram.com/pdfDocuments/mkt/download/83d0a3c8-0271-4b5a-b595-f1caa04604e5/Climate%20Sustainability%20-%20Amundi.pdf

https://cpram.com/pdfDocuments/mkt/download/1eedd3dc-fe5f-45f2-bb40-685898ee06c7/2021%20-%20Climate%20and%20%20Sustainability%20Report%20-%20Amundi.pdf

https://about.amundi.com/article/2021-4th-quarter-and-annual-results

https://about.amundi.com/article/2023-4th-quarter-and-annual-results

-> saved as sfdr_data_small.csv


## Synthetic Data - created with ChatGPT

The dataset combines verified total AuM figures from official asset-manager reports with synthetic estimates for SFDR Article 8/9 exposure. Missing values were filled using realistic assumptions based on each manager’s business model, regional footprint, and industry trends (e.g., higher SFDR shares for EU managers like Amundi and DWS; lower shares for private-markets firms such as Blackstone and KKR). All synthetic SFDR percentages and Article 8/9 AuM values are approximate and intended only for analytical or modeling purposes. They are directionally plausible but do not represent official disclosures and should not be used for regulatory reporting or investment decisions.

-> saved as: sfdr_data_synthetic.csv

### Load the Data

In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path("..")  # parent of notebooks/

sfdr = pd.read_csv(ROOT / "inputs" / "sfdr_data_small.csv")
gw = pd.read_csv(ROOT / "outputs" / "gw_scores_all.csv")

# Normalize issuer names for reliable joins
sfdr["Issuer_norm"] = sfdr["Issuer"].str.strip().str.lower()
gw["Issuer_norm"] = gw["Issuer"].str.strip().str.lower()

# Ensure percent is numeric
sfdr["Percent"] = pd.to_numeric(sfdr.get("Percent"), errors="coerce")


### Plotting the Data

In [ ]:
import matplotlib.pyplot as plt

# Keep only SFDR rows with a percent value and merge on issuer/year
sfdr_nonnull = sfdr.dropna(subset=["Percent"]).copy()
merged = gw.merge(
    sfdr_nonnull,
    on=["Issuer_norm", "Year"],
    how="inner",
    suffixes=("_gw", "_sfdr"),
)

# Choose a single issuer column for readability
if "Issuer_gw" in merged.columns and "Issuer_sfdr" in merged.columns:
    merged["Issuer"] = merged["Issuer_gw"].fillna(merged["Issuer_sfdr"])
elif "Issuer_gw" in merged.columns:
    merged["Issuer"] = merged["Issuer_gw"]
elif "Issuer_sfdr" in merged.columns:
    merged["Issuer"] = merged["Issuer_sfdr"]

# Display the matched rows
cols_to_show = ["Issuer", "Year", "GW_Risk_Score", "Percent"]
print(merged[cols_to_show])

n = len(merged)
if n >= 2:
    pearson = merged["GW_Risk_Score"].corr(merged["Percent"], method="pearson")
    spearman = merged["GW_Risk_Score"].corr(merged["Percent"], method="spearman")
    print(f"Pearson: {pearson:.3f} | Spearman: {spearman:.3f} | n={n}")
else:
    print(f"Not enough matched rows for correlation (n={n}). Add more SFDR rows to enable stats.")

# Scatter plot (works even with 1–2 points; annotate for clarity)
ax = merged.plot.scatter(
    x="GW_Risk_Score",
    y="Percent",
    title=f"GW Risk vs SFDR Percent (n={n})",
)
for _, row in merged.iterrows():
    ax.annotate(f"{row['Issuer']} {int(row['Year'])}", (row["GW_Risk_Score"], row["Percent"]))
plt.xlabel("GW Risk Score")
plt.ylabel("SFDR Article 8/9 Percent")
plt.tight_layout()
plt.show()


### Plotting the Synthetic Data

In [ ]:
ROOT = Path("..")  # parent of notebooks/

sfdr = pd.read_csv(ROOT / "inputs" / "sfdr_data_synthetic.csv")
gw = pd.read_csv(ROOT / "outputs" / "gw_scores_all.csv")

# Normalize issuer names for reliable joins
sfdr["Issuer_norm"] = sfdr["Issuer"].str.strip().str.lower()
gw["Issuer_norm"] = gw["Issuer"].str.strip().str.lower()

# Ensure percent is numeric
sfdr["Percent"] = pd.to_numeric(sfdr.get("Percent"), errors="coerce")

# Keep only SFDR rows with a percent value and merge on issuer/year
sfdr_nonnull = sfdr.dropna(subset=["Percent"]).copy()
merged = gw.merge(
    sfdr_nonnull,
    on=["Issuer_norm", "Year"],
    how="inner",
    suffixes=("_gw", "_sfdr"),
)

# Choose a single issuer column for readability
if "Issuer_gw" in merged.columns and "Issuer_sfdr" in merged.columns:
    merged["Issuer"] = merged["Issuer_gw"].fillna(merged["Issuer_sfdr"])
elif "Issuer_gw" in merged.columns:
    merged["Issuer"] = merged["Issuer_gw"]
elif "Issuer_sfdr" in merged.columns:
    merged["Issuer"] = merged["Issuer_sfdr"]

# Display the matched rows
cols_to_show = ["Issuer", "Year", "GW_Risk_Score", "Percent"]
print(merged[cols_to_show])

n = len(merged)
if n >= 2:
    pearson = merged["GW_Risk_Score"].corr(merged["Percent"], method="pearson")
    spearman = merged["GW_Risk_Score"].corr(merged["Percent"], method="spearman")
    print(f"Pearson: {pearson:.3f} | Spearman: {spearman:.3f} | n={n}")
else:
    print(f"Not enough matched rows for correlation (n={n}). Add more SFDR rows to enable stats.")

# Scatter plot (works even with 1–2 points; annotate for clarity)
ax = merged.plot.scatter(
    x="GW_Risk_Score",
    y="Percent",
    title=f"GW Risk vs SFDR Percent (n={n})",
)
for _, row in merged.iterrows():
    ax.annotate(f"{row['Issuer']} {int(row['Year'])}", (row["GW_Risk_Score"], row["Percent"]))
plt.xlabel("GW Risk Score")
plt.ylabel("SFDR Article 8/9 Percent")
plt.tight_layout()
plt.show()

### Loading all other Metrics

In [ ]:
# Correlation across multiple metrics vs SFDR Percent
metrics = ["GW_Risk_Score", "VUI_Score", "SPI_Score"]
rows = []
for m in metrics:
    if m in merged.columns:
        pearson = merged[m].corr(merged["Percent"], method="pearson") if len(merged) >= 2 else float('nan')
        spearman = merged[m].corr(merged["Percent"], method="spearman") if len(merged) >= 2 else float('nan')
        rows.append({"Metric": m, "Pearson": pearson, "Spearman": spearman, "n": len(merged)})
    else:
        rows.append({"Metric": m, "Pearson": None, "Spearman": None, "n": len(merged), "note": "not in merged"})

if rows:
    print(pd.DataFrame(rows))
else:
    print("No metrics evaluated.")


### Plots for all 3 Metrics

In [ ]:
# Scatter plots for GW, VUI, and SPI vs SFDR Percent
import matplotlib.pyplot as plt
metrics = ["GW_Risk_Score", "VUI_Score", "SPI_Score"]
fig, axes = plt.subplots(1, len(metrics), figsize=(5*len(metrics), 4), sharey=True)
if len(metrics) == 1:
    axes = [axes]
for ax, m in zip(axes, metrics):
    if m not in merged.columns:
        ax.set_visible(False)
        continue
    merged.plot.scatter(x=m, y="Percent", ax=ax, title=f"{m} vs SFDR Percent (n={len(merged)})")
    for _, row in merged.iterrows():
        ax.annotate(f"{row['Issuer']} {int(row['Year'])}", (row[m], row['Percent']))
    ax.set_xlabel(m)
axes[0].set_ylabel("SFDR Article 8/9 Percent")
plt.tight_layout()
plt.show()
